# 01 - Basic Ocean Surface Temperature

Create a CMIP7 monthly sea surface temperature file (`tos_tavg-u-hxy-sea`) on a global 10 degree latitude-longitude grid.

In [ ]:
from pathlib import Path
import json
import shutil

import cmor
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "notebooks" / "NOTEBOOKS.md").exists() or (candidate / ".git").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = start

TABLES_REPO = REPO_ROOT / "cmip7-cmor-tables"
TABLES_DIR = TABLES_REPO / "tables"
if not TABLES_DIR.exists():
    raise FileNotFoundError(f"CMIP7 tables directory not found: {TABLES_DIR}")

CV_FILE_FROM_TABLES_DIR = "../tables-cvs/cmor-cvs.json"

print(f"Using CMIP7 tables from {TABLES_REPO}")


In [ ]:
run_dir = REPO_ROOT / "notebooks" / "output" / "01_basic_ocean_surface_temperature"
if run_dir.exists():
    shutil.rmtree(run_dir)
output_dir = run_dir / "cmor_output"
output_dir.mkdir(parents=True)

DATASET_INFO = {
    "_AXIS_ENTRY_FILE": "CMIP7_coordinate.json",
    "_FORMULA_VAR_FILE": "CMIP7_formula_terms.json",
    "_cmip7_option": 1,
    "_controlled_vocabulary_file": CV_FILE_FROM_TABLES_DIR,
    "activity_id": "CMIP",
    "calendar": "360_day",
    "drs_specs": "MIP-DRS7",
    "experiment_id": "amip",
    "forcing_index": "f3",
    "frequency": "mon",
    "grid_label": "g999",
    "host_collection": "CMIP7",
    "initialization_index": "i1",
    "institution_id": "CCCma",
    "license_id": "CC-BY-4.0",
    "mip_era": "CMIP7",
    "nominal_resolution": "100 km",
    "outpath": str(output_dir),
    "physics_index": "p1",
    "realization_index": "r9",
    "region": "glb",
    "source_id": "DUMMY-MODEL",
    "tracking_prefix": "hdl:21.14107",
}
input_json = run_dir / "input.json"
input_json.write_text(json.dumps(DATASET_INFO, indent=2, sort_keys=True))

lat = np.arange(-85.0, 90.0, 10.0)
lat_bnds = np.arange(-90.0, 91.0, 10.0)
lon = np.arange(5.0, 360.0, 10.0)
lon_bnds = np.arange(0.0, 361.0, 10.0)
time = 15.0 + 30.0 * np.arange(3, dtype="f8")
time_bnds = 30.0 * np.arange(4, dtype="f8")
time_units = "days since 2018-01-01"

seasonal_cycle = np.array([0.0, 0.8, -0.4], dtype="f4")[:, None, None]
lat_term = np.sin(np.deg2rad(lat))[None, :, None] ** 2
lon_term = np.cos(np.deg2rad(lon))[None, None, :]
tos = (15.0 - 25.0 * lat_term + 1.5 * lon_term + seasonal_cycle).astype("f4")

tos.shape


In [ ]:
cmor.setup(
    inpath=str(TABLES_DIR),
    netcdf_file_action=cmor.CMOR_REPLACE,
    logfile=str(run_dir / "cmor.log"),
)
cmor.dataset_json(str(input_json))
cmor.load_table("CMIP7_ocean.json")

lat_id = cmor.axis("latitude", coord_vals=lat, cell_bounds=lat_bnds, units="degrees_north")
lon_id = cmor.axis("longitude", coord_vals=lon, cell_bounds=lon_bnds, units="degrees_east")
time_id = cmor.axis("time", coord_vals=time, cell_bounds=time_bnds, units=time_units)

variable_name = "tos_tavg-u-hxy-sea"
tos_id = cmor.variable(variable_name, "degC", [time_id, lat_id, lon_id])
compound_name = ".".join(["ocean"] + variable_name.split("_") + ["mon", "glb"])

with open(TABLES_DIR / "CMIP7_cell_measures.json") as handle:
    cell_measure = json.load(handle)["cell_measures"].get(compound_name)
if cell_measure:
    cmor.set_variable_attribute(tos_id, "cell_measures", "c", cell_measure)

with open(TABLES_DIR / "CMIP7_long_name_overrides.json") as handle:
    long_name = json.load(handle)["long_name_overrides"].get(compound_name)
if long_name:
    cmor.set_variable_attribute(tos_id, "long_name", "c", long_name)

cmor.write(tos_id, tos)
netcdf_path = Path(cmor.close(tos_id, file_name=True))
cmor.close()

netcdf_path


In [ ]:
with xr.open_dataset(netcdf_path, decode_times=False) as opened:
    ds = opened.load()
ds


In [ ]:
for name in ["mip_era", "activity_id", "source_id", "experiment_id", "variant_label", "branded_variable", "frequency", "region"]:
    print(f"{name}: {ds.attrs.get(name)}")
print(f"\nVariable shape: {ds['tos'].shape}")
print(f"Variable attributes: {dict(ds['tos'].attrs)}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
ds["tos"].isel(time=0).plot.pcolormesh(
    ax=ax,
    x="lon",
    y="lat",
    cmap="coolwarm",
    cbar_kwargs={"label": "degC"},
)
ax.set_title("CMIP7 tos, first month")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(0, 360)
ax.set_ylim(-90, 90)
ax.grid(True, alpha=0.25)
plt.show()
